In [4]:
import gdelt
import pandas as pd

gd2 = gdelt.gdelt(version=2)


Self-define events according to CAMEO root codes

In [ ]:
# search for events according to date, and return a dataframe with coverage information
events_df = gd2.Search(['2026 May 11'], table='events', coverage=True)

print(events_df.head())

   GLOBALEVENTID   SQLDATE  MonthYear  Year  FractionDate Actor1Code  \
0     1303411141  20250511     202505  2025     2025.3589        NaN   
1     1303411142  20250511     202505  2025     2025.3589        GBR   
2     1303411143  20260504     202605  2026     2026.3397        GOV   
3     1303411144  20260504     202605  2026     2026.3397        GOV   
4     1303411145  20260504     202605  2026     2026.3397        GOV   

   Actor1Name Actor1CountryCode Actor1KnownGroupCode Actor1EthnicCode  ...  \
0         NaN               NaN                  NaN              NaN  ...   
1  MANCHESTER               GBR                  NaN              NaN  ...   
2      MINIST               NaN                  NaN              NaN  ...   
3      MINIST               NaN                  NaN              NaN  ...   
4      MINIST               NaN                  NaN              NaN  ...   

  ActionGeo_Type                      ActionGeo_FullName  \
0              4   Golders Green, Barn

In [8]:
from datetime import datetime, timedelta

# define the CAMEO root codes for geopolitical events
geopolitical_root_codes = ['10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20']

# define the date range
print("downloading GDELT data...")
end_date = datetime.now()
start_date = end_date - timedelta(days=1)
date_range = pd.date_range(start_date, end_date).strftime('%Y %b %d').tolist()

# download events for the specified date range
all_events = gd2.Search(date_range, table='events', coverage=True)

if not all_events.empty:
    print(f"downloaded {len(all_events)} records from GDELT for the date range {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
    
    # geopolitical events are typically those with EventCode starting with 1x, so we can filter based on that
    all_events['EventRootCode'] = all_events['EventCode'].astype(str).str[:2]
    
    geo_events = all_events[all_events['EventRootCode'].isin(geopolitical_root_codes)]
    print(f"geopolitical events: {len(geo_events)}")
    
    key_columns = ['SQLDATE', 'Actor1Name', 'Actor2Name', 'EventCode', 
                   'EventRootCode', 'GoldsteinScale', 'AvgTone', 'ActionGeo_FullName']
    
    display_columns = [col for col in key_columns if col in geo_events.columns]
    
    print("\n===== Geopolitical Events Example =====")
    print(geo_events[display_columns].head(10).to_string())
    
    # summarize event type distribution
    event_counts = geo_events['EventRootCode'].value_counts()
    print("\n===== Event Type Distribution =====")
    for code, count in event_counts.items():
        # Simple CAMEO explanation
        cameo_names = {
            '10': 'Demand', '11': 'Disapprove', '12': 'Reject',
            '13': 'Threaten', '14': 'Protest', '15': 'Force Posture',
            '16': 'Reduce Relations', '17': 'Coerce', 
            '18': 'Assault', '19': 'Fight', '20': 'Unconventional Mass Violence'
        }
        name = cameo_names.get(code, 'Unknown')
        print(f"{code} - {name}: {count} records")
    
    # save to CSV
    output_file = f"geopolitical_events_{datetime.now().strftime('%Y%m%d')}.csv"
    geo_events.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\nData saved to: {output_file}")
    
else:
    print("No data retrieved. Please check your network connection or the date range.")

downloading GDELT data...
downloaded 76894 records from GDELT for the date range 2026-05-10 to 2026-05-11
geopolitical events: 19960

===== Geopolitical Events Example =====
     SQLDATE  Actor1Name        Actor2Name EventCode EventRootCode  GoldsteinScale    AvgTone                      ActionGeo_FullName
0   20250510         NaN            JEWISH       190            19           -10.0  -6.007752   Golders Green, Barnet, United Kingdom
1   20250510         NaN            JEWISH       190            19           -10.0  -6.007752  Manchester, Manchester, United Kingdom
2   20250510  MANCHESTER            JEWISH       190            19           -10.0  -6.007752  Manchester, Manchester, United Kingdom
32  20260510         NaN          INDUSTRY       190            19           -10.0  -4.971751                           United States
36  20260510         NaN          DEPUTIES      1124            11            -2.0  -6.818182                    Texas, United States
37  20260510         N

Find related events according to keywords & GLOBALEVENTID

In [ ]:
"""find events(GLOBALEVENTID) by keyword"""

print("downloading...")
events_df = gd2.Search(['2026 May 10'], table='events', coverage=True)
print(f"downloaded {len(events_df)} records")

def search_events_by_keyword(df, keyword, column='Actor1Name'):
    """search in the specified column for the keyword and return the event IDs and basic information"""
    mask = df[column].astype(str).str.contains(keyword, case=False, na=False)
    results = df[mask][['GLOBALEVENTID', 'SQLDATE', 'Actor1Name', 'Actor2Name', 
                         'EventCode', 'ActionGeo_FullName', 'AvgTone']]
    return results.drop_duplicates(subset='GLOBALEVENTID')

# example: search for events involving "USA"
print("\n===== example: search for events involving \"USA\" =====")
usa_events = search_events_by_keyword(events_df, 'USA', 'Actor1Name')
print(f"found {len(usa_events)} events")
print(usa_events.head(10).to_string())


downloading...
downloaded 64629 records

===== example: search for events involving "USA" =====
found 48 events
       GLOBALEVENTID   SQLDATE  Actor1Name      Actor2Name EventCode                                ActionGeo_FullName   AvgTone
4102      1303294579  20260510       BUSAN         CHINESE       040                                             China -1.971831
4103      1303294580  20260510       BUSAN         CHINESE       040                                             China -1.971831
4997      1303312715  20260510   JERUSALEM          ISRAEL       192         Wadi Qana, West Bank (general), West Bank -7.530120
5003      1303312721  20260510   JERUSALEM       WEST BANK       042               Jerusalem, Israel (general), Israel -7.530120
5004      1303312722  20260510   JERUSALEM     PALESTINIAN       192               Jerusalem, Israel (general), Israel -7.530120
5005      1303312723  20260510   JERUSALEM     PALESTINIAN       192         Wadi Qana, West Bank (general), West 

In [19]:
"""search for all articles related to a specific event ID"""

def get_event_news(df, event_id):
    event_data = df[df['GLOBALEVENTID'] == event_id]
    if len(event_data) == 0:
        return None
    return event_data[['GLOBALEVENTID', 'SQLDATE', 'Actor1Name', 'Actor2Name', 
                        'EventCode', 'NumArticles', 'AvgTone', 'SOURCEURL']]

# example: use the first found ID for demonstration
if len(usa_events) > 0:
    sample_id = usa_events.iloc[0]['GLOBALEVENTID']
    print(f"\n===== all articles for event ID: {sample_id} =====")
    news = get_event_news(events_df, sample_id)
    if news is not None:
        print(f"found {len(news)} records (from different sources)")
        print(f"total articles: {news['NumArticles'].sum()}")
        for i, row in news.head(5).iterrows():
            print(f"  source: {row['SOURCEURL'][:80]}...")
            print(f"  tone: {row['AvgTone']}")


===== all articles for event ID: 1303294579 =====
found 1 records (from different sources)
total articles: 4
  source: https://www.dailykos.com/stories/2026/5/9/800035133/series/trump-economy-broken-...
  tone: -1.9718309859155
